#Bài 1

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Tạo bảng
cursor.execute("""
CREATE TABLE student (
    student_id INTEGER,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
)
""")

cursor.execute("""
CREATE TABLE course (
    id INTEGER,
    course_name TEXT
)
""")

# Insert dữ liệu
students = [
    (1, "Nguyen Minh Hoang", "May Tinh", 12, 6.7),
    (2, "Tran Thi Lan", "Kinh Te", 34, 9.2),
    (3, "Pham Van Nam", "Toan Tin", None, 7.9),
    (4, "Le Thanh Huyen", "Toan Tin", 20, 7.2),
    (5, "Vu Quoc Anh", "May Tinh", 24, 8.0),
    (6, "Dang Thuy Linh", "May Tinh", 24, 5.5),
    (7, "Bui Tien Dung", "Kinh Te", 34, 9.2),
    (8, "Ho Ngoc Mai", "Toan Tin", 20, 8.8),
    (9, "Duong Huu Phuc", "Kinh Te", None, 7.2),
    (10, "Cao Thi Hanh", "May Tinh", None, 7.0)
]

courses = [
    (12, "Giai tich"),
    (34, "Thong ke"),
    (26, "Tin hoc")
]

cursor.executemany("INSERT INTO student VALUES (?, ?, ?, ?, ?)", students)
cursor.executemany("INSERT INTO course VALUES (?, ?)", courses)
conn.commit()

# CROSS JOIN
print("CROSS JOIN")
print(pd.read_sql_query("SELECT * FROM student CROSS JOIN course", conn))

# INNER JOIN
print("\nINNER JOIN")
print(pd.read_sql_query("""
SELECT * FROM student s
JOIN course c ON s.course_id = c.id
""", conn))

# LEFT JOIN
print("\nLEFT JOIN")
print(pd.read_sql_query("""
SELECT * FROM student s
LEFT JOIN course c ON s.course_id = c.id
""", conn))

# RIGHT JOIN (giả lập)
print("\nRIGHT JOIN (fake)")
print(pd.read_sql_query("""
SELECT * FROM course c
LEFT JOIN student s ON s.course_id = c.id
""", conn))

CROSS JOIN
    student_id               name     class  course_id  score  id course_name
0            1  Nguyen Minh Hoang  May Tinh       12.0    6.7  12   Giai tich
1            1  Nguyen Minh Hoang  May Tinh       12.0    6.7  34    Thong ke
2            1  Nguyen Minh Hoang  May Tinh       12.0    6.7  26     Tin hoc
3            2       Tran Thi Lan   Kinh Te       34.0    9.2  12   Giai tich
4            2       Tran Thi Lan   Kinh Te       34.0    9.2  34    Thong ke
5            2       Tran Thi Lan   Kinh Te       34.0    9.2  26     Tin hoc
6            3       Pham Van Nam  Toan Tin        NaN    7.9  12   Giai tich
7            3       Pham Van Nam  Toan Tin        NaN    7.9  34    Thong ke
8            3       Pham Van Nam  Toan Tin        NaN    7.9  26     Tin hoc
9            4     Le Thanh Huyen  Toan Tin       20.0    7.2  12   Giai tich
10           4     Le Thanh Huyen  Toan Tin       20.0    7.2  34    Thong ke
11           4     Le Thanh Huyen  Toan Tin       20.

#Bài 2

In [2]:
# Xóa course_id không hợp lệ
cursor.execute("""
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course)
""")
conn.commit()

# a. Theo lớp
print("Theo lớp")
print(pd.read_sql_query("""
SELECT class, COUNT(*) AS total, AVG(score) AS avg_score
FROM student
GROUP BY class
""", conn))

# b. Theo môn
print("\nTheo môn")
print(pd.read_sql_query("""
SELECT c.course_name, COUNT(*) AS total, AVG(s.score) AS avg_score
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""", conn))

# c. Phân loại
print("\nPhân loại")
print(pd.read_sql_query("""
SELECT c.course_name,
       AVG(s.score) AS avg_score,
       CASE
           WHEN AVG(s.score) >= 9 THEN 'Xuat sac'
           WHEN AVG(s.score) >= 6 THEN 'Tot'
           ELSE 'Kem'
       END AS classification
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""", conn))

Theo lớp
      class  total  avg_score
0   Kinh Te      3   8.533333
1  May Tinh      2   6.850000
2  Toan Tin      1   7.900000

Theo môn
  course_name  total  avg_score
0   Giai tich      1        6.7
1    Thong ke      2        9.2

Phân loại
  course_name  avg_score classification
0   Giai tich        6.7            Tot
1    Thong ke        9.2       Xuat sac


#Bài 3

In [3]:
# a. Theo điểm
print("Ranking toàn bộ")
print(pd.read_sql_query("""
SELECT *, RANK() OVER (ORDER BY score DESC) AS rank
FROM student
""", conn))

# b. Theo lớp
print("\nRanking theo lớp")
print(pd.read_sql_query("""
SELECT *, RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank
FROM student
""", conn))

# c. Theo môn
print("\nRanking theo môn")
print(pd.read_sql_query("""
SELECT s.*, c.course_name,
       RANK() OVER (PARTITION BY s.course_id ORDER BY score DESC) AS rank
FROM student s
JOIN course c ON s.course_id = c.id
""", conn))

# Top 3 cao & thấp
print("\nTop 3 cao")
print(pd.read_sql_query("SELECT * FROM student ORDER BY score DESC LIMIT 3", conn))

print("\nTop 3 thấp")
print(pd.read_sql_query("SELECT * FROM student ORDER BY score ASC LIMIT 3", conn))

Ranking toàn bộ
   student_id               name     class  course_id  score  rank
0           2       Tran Thi Lan   Kinh Te       34.0    9.2     1
1           7      Bui Tien Dung   Kinh Te       34.0    9.2     1
2           3       Pham Van Nam  Toan Tin        NaN    7.9     3
3           9     Duong Huu Phuc   Kinh Te        NaN    7.2     4
4          10       Cao Thi Hanh  May Tinh        NaN    7.0     5
5           1  Nguyen Minh Hoang  May Tinh       12.0    6.7     6

Ranking theo lớp
   student_id               name     class  course_id  score  rank
0           2       Tran Thi Lan   Kinh Te       34.0    9.2     1
1           7      Bui Tien Dung   Kinh Te       34.0    9.2     1
2           9     Duong Huu Phuc   Kinh Te        NaN    7.2     3
3          10       Cao Thi Hanh  May Tinh        NaN    7.0     1
4           1  Nguyen Minh Hoang  May Tinh       12.0    6.7     2
5           3       Pham Van Nam  Toan Tin        NaN    7.9     1

Ranking theo môn
   student

#Bài 4

In [4]:
# Thêm cột
cursor.execute("ALTER TABLE student ADD COLUMN graduation_date TEXT")

# Cập nhật theo rank (điểm cao -> tốt nghiệp sớm)
cursor.execute("""
UPDATE student
SET graduation_date = datetime('now', '+' || (
    SELECT COUNT(*)
    FROM student s2
    WHERE s2.score > student.score
) || ' days')
""")

conn.commit()

# Kết quả
print(pd.read_sql_query("SELECT * FROM student", conn))

   student_id               name     class  course_id  score  \
0           1  Nguyen Minh Hoang  May Tinh       12.0    6.7   
1           2       Tran Thi Lan   Kinh Te       34.0    9.2   
2           3       Pham Van Nam  Toan Tin        NaN    7.9   
3           7      Bui Tien Dung   Kinh Te       34.0    9.2   
4           9     Duong Huu Phuc   Kinh Te        NaN    7.2   
5          10       Cao Thi Hanh  May Tinh        NaN    7.0   

       graduation_date  
0  2026-04-08 08:16:01  
1  2026-04-03 08:16:01  
2  2026-04-05 08:16:01  
3  2026-04-03 08:16:01  
4  2026-04-06 08:16:01  
5  2026-04-07 08:16:01  
